# Frequency calibration workflow

This notebook focuses on the two frequency-calibration loops you will use
most often after initial bring-up: qubit control frequency and readout
frequency. Adjust the setup cell for your system before running it.


## 1. Create an `Experiment`

Start from one qubit whose readout and drive channels are already wired
and visible in the configuration.


In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    qubits=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

Q0 = exp.qubit_labels[0]
print("target qubit:", Q0)

## 2. Connect and collect a baseline

A waveform check plus one Rabi measurement is a good baseline before you
move the frequencies.


In [ ]:
exp.connect()

waveform_result = exp.check_waveform(targets=Q0, n_shots=512, plot=True)
rabi_result = exp.obtain_rabi_params(
    targets=Q0,
    time_range=np.arange(16, 321, 16),
    n_shots=512,
    plot=True,
)

## 3. Calibrate the control frequency

Sweep a detuning window around the current qubit frequency and let Qubex
fit the updated resonance.


In [ ]:
control_frequency_result = exp.calibrate_control_frequency(
    targets=Q0,
    detuning_range=np.linspace(-0.05, 0.05, 41),
    time_range=np.arange(32, 257, 32),
    n_shots=512,
    plot=True,
)

## 4. Re-check the Rabi oscillation

Re-running `obtain_rabi_params()` immediately shows whether the updated
qubit frequency moved the oscillation closer to the expected model.


In [ ]:
rabi_after_control = exp.obtain_rabi_params(
    targets=Q0,
    time_range=np.arange(16, 321, 16),
    n_shots=512,
    plot=True,
)

## 5. Calibrate the readout frequency

Use a narrower detuning window once the qubit drive side is in place.


In [ ]:
readout_frequency_result = exp.calibrate_readout_frequency(
    targets=Q0,
    detuning_range=np.linspace(-0.010, 0.010, 41),
    n_shots=512,
    plot=True,
)

## 6. Optionally persist and reload

Use this when you want later notebooks to start from the newly fitted
frequencies.


In [ ]:
# exp.calib_note.save()
# exp.reload()
